Definition Function

In [10]:
# Parse string to point of polygon
def parse_polygon(poly_str):

    matches = re.findall(
        r'\[(\-?\d+),(\-?\d+)\]',
        str(poly_str)
    )

    points = [
        (float(x), float(y))
        for x, y in matches
    ]

    # fechar polígono
    if points[0] != points[-1]:
        points.append(points[0])

    return points

def polygon_orientation(poly):
    """
    Calcula orientação dominante do polígono
    usando PCA.

    Returns
    -------
    angle_deg : float
        orientação dominante [0,180]

    anisotropy : float
        confiança da orientação
    """

    coords = np.array(poly.exterior.coords)

    # PCA
    pca = PCA(n_components=2)
    pca.fit(coords)

    # eixo principal
    principal_vector = pca.components_[0]

    # ângulo
    angle = np.degrees(
        np.arctan2(
            principal_vector[1],
            principal_vector[0]
        )
    )

    # normalizar para [0,180]
    angle = angle % 180

    # razão dos autovalores
    anisotropy = (
        pca.explained_variance_[0]
        /
        pca.explained_variance_[1]
    )

    return angle, anisotropy

Prepare data for training

In [11]:
import shapely
import pandas as pd
import re
import numpy as np

from sklearn.preprocessing import LabelEncoder
from shapely.geometry import Polygon
from sklearn.decomposition import PCA

df = pd.read_csv("consolidado_metricas_2016_2025.csv")

# Split Anomaly ID
df['Figura_name'] =  df['AnomalyID'].str.split('#', expand=True)[0]
df['ID_Figura'] =  df['AnomalyID'].str.split('#', expand=True)[1]

# Stay only corrosion
df = df[(df['FacadeRoofID'] != 'F?') & (df['AnomalyType'] == 'corrosion')].copy()

# convert string coordenates to number
df['polygon_points'] = (df['Coordinates (x,y)'].apply(parse_polygon))

# Create geometry of polygon
df["geometry"] = (df["polygon_points"].apply(Polygon))

# Calculate centroid
df['centroid'] = shapely.centroid(df['geometry'])

# Split centroid coords
df['centroid_x'] = df['centroid'].apply(lambda p: p.x)
df['centroid_y'] = df['centroid'].apply(lambda p: p.y)

# Calculate angle
results = (df["geometry"].apply(polygon_orientation))

# define data degree angle
df[["orientation_deg", "anisotropy"]] = pd.DataFrame(results.tolist(), index=df.index)
df['sin_2theta'] = np.sin(np.radians(2 * df['orientation_deg']))
df['cos_2theta'] = np.cos(np.radians(2 * df['orientation_deg']))

# Order data
df_balanced = df.sort_values(by=['Figura_name', 'ID_Figura', 'data'], ascending=[True,True,True])

# Make delta (lag features)
df_balanced['centroid_x_delta'] = df_balanced.groupby('Figura_name')['centroid_x'].diff()
df_balanced['centroid_y_delta'] = df_balanced.groupby('Figura_name')['centroid_y'].diff()
df_balanced['orientation_deg_delta'] = df_balanced.groupby('Figura_name')['orientation_deg'].diff()
df_balanced['sin_2theta_delta'] = df_balanced.groupby('Figura_name')['sin_2theta'].diff()
df_balanced['cos_2theta_delta'] = df_balanced.groupby('Figura_name')['cos_2theta'].diff()

# Direção real para onde a corrosão está se movendo
df_balanced['movement_angle'] = np.degrees(np.arctan2(df_balanced['centroid_y_delta'], df_balanced['centroid_x_delta']))
df_balanced['movement_angle'] = df_balanced['movement_angle'].fillna(0)
df_balanced['movement_sin'] = np.sin(np.radians(df_balanced['movement_angle']))
df_balanced['movement_cos'] = np.cos(np.radians(df_balanced['movement_angle']))

# Fill NaN values
df_balanced['centroid_x_delta'] = df_balanced['centroid_x_delta'].fillna(0)
df_balanced['centroid_y_delta'] = df_balanced['centroid_y_delta'].fillna(0)
df_balanced['orientation_deg_delta'] = df_balanced['orientation_deg_delta'].fillna(0)
df_balanced['cos_2theta_delta'] = df_balanced['cos_2theta_delta'].fillna(0)
df_balanced['sin_2theta_delta'] = df_balanced['sin_2theta_delta'].fillna(0)

# Encoding Categorical data
le_facade = LabelEncoder()
le_anomaly = LabelEncoder()
le_figura = LabelEncoder()

df_balanced['FacadeRoofID'] = le_facade.fit_transform(df_balanced['FacadeRoofID'])
df_balanced['AnomalyType'] = le_anomaly.fit_transform(df_balanced['AnomalyType'])
df_balanced['Figura_name'] = le_figura.fit_transform(df_balanced['Figura_name'])

# Drop unused fields
df_balanced.drop('ImagePath', axis=1, inplace=True)
df_balanced.drop('Coordinates', axis=1, inplace=True)
df_balanced.drop('Area (cm2)', axis=1, inplace=True)
df_balanced.drop('AnomalyID', axis=1, inplace=True)
df_balanced.drop('Coordinates (x,y)', axis=1, inplace=True)
df_balanced.drop('polygon_points', axis=1, inplace=True)
df_balanced.drop('centroid', axis=1, inplace=True)
df_balanced.drop('geometry', axis=1, inplace=True)
df_balanced.drop('data', axis=1, inplace=True)
df_balanced.drop('SimulationDate', axis=1, inplace=True)
df_balanced.drop('Surface3DArea_cm2', axis=1, inplace=True)
df_balanced.drop('Added3DArea_cm2', axis=1, inplace=True)
df_balanced.drop('SurfaceVertexCount', axis=1, inplace=True)
df_balanced.drop('SurfaceNormalX', axis=1, inplace=True)
df_balanced.drop('SurfaceNormalY', axis=1, inplace=True)
df_balanced.drop('SurfaceNormalZ', axis=1, inplace=True)
df_balanced.drop('SurfaceAzimuthDeg', axis=1, inplace=True)
df_balanced.drop('WindDirectionDeg', axis=1, inplace=True)
df_balanced.drop('WDRDirectionDeg', axis=1, inplace=True)
df_balanced.drop('ProximityFactor', axis=1, inplace=True)
df_balanced.drop('DistanceWater_m', axis=1, inplace=True)
df_balanced.drop('DistanceDamage_m', axis=1, inplace=True)
df_balanced.drop('CorrosionNeighbors', axis=1, inplace=True)
df_balanced.drop('RelativeGrowthTarget', axis=1, inplace=True)
df_balanced.drop('TemporalGrowthFactor', axis=1, inplace=True)
df_balanced.drop('OriginalShapeAngleDeg', axis=1, inplace=True)
df_balanced.drop('OriginalElongationRatio', axis=1, inplace=True)
df_balanced.drop('ShapeTrendStrength', axis=1, inplace=True)
df_balanced.drop('FinalGrowthAngleDeg', axis=1, inplace=True)
df_balanced.drop('GeometrySupportCoverage', axis=1, inplace=True)
df_balanced.drop('MorphIterations', axis=1, inplace=True)
df_balanced.drop('MeanFrontierExposure', axis=1, inplace=True)
df_balanced.drop('AddedVertices3D', axis=1, inplace=True)
df_balanced.drop('MeanTransitionUsedDeg', axis=1, inplace=True)
df_balanced.drop('MaxTransitionUsedDeg', axis=1, inplace=True)
df_balanced.drop('SurfacePatchVertices', axis=1, inplace=True)
df_balanced.drop('SurfaceSpacing_m', axis=1, inplace=True)
df_balanced.drop('SurfaceGrowthMethod', axis=1, inplace=True)

df_balanced.to_csv('dataset/Coords_data_training.csv', index=False)


Prepare data for Test

In [12]:
df = pd.read_csv("obra/2026/01_01_2026/pred/ps/metricas_anomalias_faces_malha_2026-01-01_ambiente_3D_HIBRIDO.csv")

# Split Anomaly ID
df['Figura_name'] =  df['AnomalyID'].str.split('#', expand=True)[0]
df['ID_Figura'] =  df['AnomalyID'].str.split('#', expand=True)[1]

# Stay only corrosion
df = df[(df['FacadeRoofID'] != 'F?') & (df['AnomalyType'] == 'corrosion')].copy()

# convert string coordenates to number
df['polygon_points'] = (df['Coordinates (x,y)'].apply(parse_polygon))

# Create geometry of polygon
df["geometry"] = (df["polygon_points"].apply(Polygon))

# Calculate centroid
df['centroid'] = shapely.centroid(df['geometry'])

# Split centroid coords
df['centroid_x'] = df['centroid'].apply(lambda p: p.x)
df['centroid_y'] = df['centroid'].apply(lambda p: p.y)

# Calculate angle
results = (df["geometry"].apply(polygon_orientation))

# define data degree angle
df[["orientation_deg", "anisotropy"]] = pd.DataFrame(results.tolist(), index=df.index)
df['sin_2theta'] = np.sin(np.radians(2 * df['orientation_deg']))
df['cos_2theta'] = np.cos(np.radians(2 * df['orientation_deg']))

# Order data
df_balanced = df.sort_values(by=['Figura_name', 'ID_Figura', 'data'], ascending=[True,True,True])

# Make delta (lag features)
df_balanced['centroid_x_delta'] = df_balanced.groupby('Figura_name')['centroid_x'].diff()
df_balanced['centroid_y_delta'] = df_balanced.groupby('Figura_name')['centroid_y'].diff()
df_balanced['orientation_deg_delta'] = df_balanced.groupby('Figura_name')['orientation_deg'].diff()
df_balanced['sin_2theta_delta'] = df_balanced.groupby('Figura_name')['sin_2theta'].diff()
df_balanced['cos_2theta_delta'] = df_balanced.groupby('Figura_name')['cos_2theta'].diff()

# Direção real para onde a corrosão está se movendo
df_balanced['movement_angle'] = np.degrees(np.arctan2(df_balanced['centroid_y_delta'], df_balanced['centroid_x_delta']))
df_balanced['movement_sin'] = np.sin(np.radians(df_balanced['movement_angle']))
df_balanced['movement_cos'] = np.cos(np.radians(df_balanced['movement_angle']))

# Fill NaN values
df_balanced['centroid_x_delta'] = df_balanced['centroid_x_delta'].fillna(0)
df_balanced['centroid_y_delta'] = df_balanced['centroid_y_delta'].fillna(0)
df_balanced['orientation_deg_delta'] = df_balanced['orientation_deg_delta'].fillna(0)
df_balanced['cos_2theta_delta'] = df_balanced['cos_2theta_delta'].fillna(0)
df_balanced['sin_2theta_delta'] = df_balanced['sin_2theta_delta'].fillna(0)
df_balanced['movement_angle'] = df_balanced['movement_angle'].fillna(0)

# Encoding Categorical data
df_balanced['FacadeRoofID'] = le_facade.transform(df_balanced['FacadeRoofID'])
df_balanced['AnomalyType'] = le_anomaly.transform(df_balanced['AnomalyType'])
df_balanced['Figura_name'] = le_figura.transform(df_balanced['Figura_name'])

# Drop unused fields
df_balanced.drop('ImagePath', axis=1, inplace=True)
df_balanced.drop('Coordinates', axis=1, inplace=True)
df_balanced.drop('Area (cm2)', axis=1, inplace=True)
df_balanced.drop('AnomalyID', axis=1, inplace=True)
df_balanced.drop('Coordinates (x,y)', axis=1, inplace=True)
df_balanced.drop('polygon_points', axis=1, inplace=True)
df_balanced.drop('centroid', axis=1, inplace=True)
df_balanced.drop('geometry', axis=1, inplace=True)
df_balanced.drop('data', axis=1, inplace=True)
df_balanced.drop('SimulationDate', axis=1, inplace=True)
df_balanced.drop('Surface3DArea_cm2', axis=1, inplace=True)
df_balanced.drop('Added3DArea_cm2', axis=1, inplace=True)
df_balanced.drop('SurfaceVertexCount', axis=1, inplace=True)
df_balanced.drop('SurfaceNormalX', axis=1, inplace=True)
df_balanced.drop('SurfaceNormalY', axis=1, inplace=True)
df_balanced.drop('SurfaceNormalZ', axis=1, inplace=True)
df_balanced.drop('SurfaceAzimuthDeg', axis=1, inplace=True)
df_balanced.drop('WindDirectionDeg', axis=1, inplace=True)
df_balanced.drop('WDRDirectionDeg', axis=1, inplace=True)
df_balanced.drop('ProximityFactor', axis=1, inplace=True)
df_balanced.drop('DistanceWater_m', axis=1, inplace=True)
df_balanced.drop('DistanceDamage_m', axis=1, inplace=True)
df_balanced.drop('CorrosionNeighbors', axis=1, inplace=True)
df_balanced.drop('RelativeGrowthTarget', axis=1, inplace=True)
df_balanced.drop('TemporalGrowthFactor', axis=1, inplace=True)
df_balanced.drop('OriginalShapeAngleDeg', axis=1, inplace=True)
df_balanced.drop('OriginalElongationRatio', axis=1, inplace=True)
df_balanced.drop('ShapeTrendStrength', axis=1, inplace=True)
df_balanced.drop('FinalGrowthAngleDeg', axis=1, inplace=True)
df_balanced.drop('GeometrySupportCoverage', axis=1, inplace=True)
df_balanced.drop('MorphIterations', axis=1, inplace=True)
df_balanced.drop('MeanFrontierExposure', axis=1, inplace=True)
df_balanced.drop('AddedVertices3D', axis=1, inplace=True)
df_balanced.drop('MeanTransitionUsedDeg', axis=1, inplace=True)
df_balanced.drop('MaxTransitionUsedDeg', axis=1, inplace=True)
df_balanced.drop('SurfacePatchVertices', axis=1, inplace=True)
df_balanced.drop('SurfaceSpacing_m', axis=1, inplace=True)
df_balanced.drop('SurfaceGrowthMethod', axis=1, inplace=True)

df_balanced.to_csv('dataset/Coords_data_test.csv', index=False)
